<a href="https://colab.research.google.com/github/minju0236/Hankyung-Bootcamp/blob/main/Day9_10_a_(200416)_Mutation_%26_Query_Invalidation_%26_Stale_While_Revalidate_%EC%8B%A4%EC%8A%B5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# React Query 기반 금융 거래 Mutation 및 Optimistic Update

In [30]:
%%writefile react-query-practice/server/server.js

const express = require('express');
const cors = require('cors');
const mysql = require('mysql2/promise');
const path = require('path');

const app = express();
const PORT = 3000;

app.use(cors());
app.use(express.json());

// static 디렉토리 제공
app.use(express.static(path.join(__dirname, 'static')));

// DB 연결
const pool = mysql.createPool({
  host: 'localhost',
  user: 'testuser',
  password: '1234',
  database: 'testdb'
});

// 거래 조회
app.get('/api/transfers', async function (req, res) {
  const [rows] = await pool.query(
    'SELECT * FROM transfers ORDER BY id DESC'
  );
  res.json(rows);
});

// 거래 등록 (Mutation)
app.post('/api/transfers', async function (req, res) {
  const { sender_name, receiver_name, amount, transfer_type } = req.body;

  if (amount >= 1000000) {
    return res.status(500).json({ message: '거래 실패' });
  }

  await pool.query(
    'INSERT INTO transfers (sender_name, receiver_name, amount, transfer_type, status) VALUES (?, ?, ?, ?, ?)',
    [sender_name, receiver_name, amount, transfer_type, '완료']
  );

  res.json({ success: true });
});

// React 라우팅 대응
app.get('/', function (req, res) {
  res.sendFile(path.join(__dirname, 'static', 'index.html'));
});

app.get('/*rest', function (req, res) {
  res.sendFile(path.join(__dirname, 'static', 'index.html'));
});

app.listen(PORT, function () {
  console.log('server running on 3000');
});

Overwriting react-query-practice/server/server.js


In [38]:
%%writefile react-query-practice/vite/src/main.jsx

import React from 'react';
import ReactDOM from 'react-dom/client';
import { QueryClient, QueryClientProvider } from '@tanstack/react-query';
import App from './App';
import './styles/app.css';

const queryClient = new QueryClient();

ReactDOM.createRoot(document.getElementById('root')).render(
  <React.StrictMode>
    <QueryClientProvider client={queryClient}>
      <App />
    </QueryClientProvider>
  </React.StrictMode>
);

Overwriting react-query-practice/vite/src/main.jsx


In [32]:
%%writefile react-query-practice/vite/src/App.jsx

import { useQuery } from '@tanstack/react-query';
import { fetchTransfers } from './api/transfersApi';
import TransferForm from './components/TransferForm';
import TransferSummary from './components/TransferSummary';
import TransferList from './components/TransferList';

function App() {
  const { data, isLoading, isFetching, error } = useQuery({
    queryKey: ['transfers'],
    queryFn: fetchTransfers,
    staleTime: 5000
  });

  return (
    <div className="container">
      <h1>금융 거래 등록 실습</h1>
      <p className="description">
        React Query의 Mutation과 Optimistic Update 동작을 확인한다.
      </p>

      <TransferForm />

      <div className="panel">
        {isLoading && <p className="status">최초 로딩 중입니다.</p>}
        {!isLoading && isFetching && (
          <p className="status">최신 데이터를 확인하는 중입니다.</p>
        )}
        {error && <p className="status error">{error.message}</p>}
      </div>

      {data && (
        <>
          <TransferSummary transfers={data} />
          <TransferList transfers={data} />
        </>
      )}
    </div>
  );
}

export default App;

Overwriting react-query-practice/vite/src/App.jsx


In [33]:
%%writefile react-query-practice/vite/src/api/transfersApi.js

export async function fetchTransfers() {
  const response = await fetch('/api/transfers');

  if (!response.ok) {
    throw new Error('거래 목록 조회 실패');
  }

  return await response.json();
}

export async function createTransfer(transferData) {
  const response = await fetch('/api/transfers', {
    method: 'POST',
    headers: {
      'Content-Type': 'application/json'
    },
    body: JSON.stringify(transferData)
  });

  if (!response.ok) {
    const errorData = await response.json();
    throw new Error(errorData.message || '거래 등록 실패');
  }

  return await response.json();
}

Overwriting react-query-practice/vite/src/api/transfersApi.js


In [34]:
%%writefile react-query-practice/vite/src/components/TransferForm.jsx

import { useState } from 'react';
import { useMutation, useQueryClient } from '@tanstack/react-query';
import { createTransfer } from '../api/transfersApi';

function TransferForm() {
  const queryClient = useQueryClient();

  const [form, setForm] = useState({
    sender_name: '',
    receiver_name: '',
    amount: '',
    transfer_type: '계좌이체'
  });

  const mutation = useMutation({
    mutationFn: createTransfer,

    onMutate: async function (newData) {
      await queryClient.cancelQueries({ queryKey: ['transfers'] });

      const previousTransfers = queryClient.getQueryData(['transfers']);

      queryClient.setQueryData(['transfers'], function (old) {
        return [
          {
            id: Date.now(),
            sender_name: newData.sender_name,
            receiver_name: newData.receiver_name,
            amount: newData.amount,
            transfer_type: newData.transfer_type,
            status: '처리중',
            created_at: '요청 중'
          },
          ...(old || []) //기존 데이터가 없으면 빈배열을 복사해라
        ];
      });

      return { previousTransfers };
    },

    onError: function (error, newData, context) {
      queryClient.setQueryData(['transfers'], context.previousTransfers);
    },

    onSettled: function () {
      queryClient.invalidateQueries({ queryKey: ['transfers'] });
    }
  });

  function handleChange(e) {
    setForm({
      ...form,
      [e.target.name]: e.target.value
    });
  }

  function handleSubmit(e) {
    e.preventDefault();

    mutation.mutate({
      ...form,
      amount: Number(form.amount)
    });

    setForm({
      sender_name: '',
      receiver_name: '',
      amount: '',
      transfer_type: '계좌이체'
    });
  }

  return (
    <div className="panel">
      <h2>거래 등록</h2>

      <form onSubmit={handleSubmit} className="transfer-form">
        <input
          type="text"
          name="sender_name"
          value={form.sender_name}
          onChange={handleChange}
          placeholder="보내는 사람"
        />

        <input
          type="text"
          name="receiver_name"
          value={form.receiver_name}
          onChange={handleChange}
          placeholder="받는 사람"
        />

        <input
          type="number"
          name="amount"
          value={form.amount}
          onChange={handleChange}
          placeholder="금액"
        />

        <select
          name="transfer_type"
          value={form.transfer_type}
          onChange={handleChange}
        >
          <option value="계좌이체">계좌이체</option>
          <option value="자동이체">자동이체</option>
          <option value="예약이체">예약이체</option>
        </select>

        <button type="submit" className="action-button">
          거래 등록
        </button>
      </form>

      {mutation.isPending && (
        <p className="status">서버에 거래를 등록하는 중입니다.</p>
      )}

      {mutation.isError && (
        <p className="status error">{mutation.error.message}</p>
      )}
    </div>
  );
}

export default TransferForm;

Overwriting react-query-practice/vite/src/components/TransferForm.jsx


In [35]:
%%writefile react-query-practice/vite/src/components/TransferList.jsx

function TransferList({ transfers }) {
  return (
    <div className="panel">
      <h2>거래 목록</h2>

      <table className="transfer-table">
        <thead>
          <tr>
            <th>번호</th>
            <th>보내는 사람</th>
            <th>받는 사람</th>
            <th>금액</th>
            <th>이체 유형</th>
            <th>상태</th>
            <th>등록 시각</th>
          </tr>
        </thead>
        <tbody>
          {transfers.map(function (item) {
            return (
              <tr key={item.id}>
                <td>{item.id}</td>
                <td>{item.sender_name}</td>
                <td>{item.receiver_name}</td>
                <td>{Number(item.amount).toLocaleString('ko-KR')}원</td>
                <td>{item.transfer_type}</td>
                <td>{item.status}</td>
                <td>{item.created_at}</td>
              </tr>
            );
          })}
        </tbody>
      </table>
    </div>
  );
}

export default TransferList;

Overwriting react-query-practice/vite/src/components/TransferList.jsx


In [36]:
%%writefile react-query-practice/vite/src/components/TransferSummary.jsx

function TransferSummary({ transfers }) {
  const totalCount = transfers.length;

  const totalAmount = transfers.reduce(function (sum, item) {
    return sum + Number(item.amount);
  }, 0);

  return (
    <div className="panel">
      <h2>거래 요약</h2>
      <p>총 거래 건수: {totalCount}건</p>
      <p>총 거래 금액: {totalAmount.toLocaleString('ko-KR')}원</p>
    </div>
  );
}

export default TransferSummary;

Overwriting react-query-practice/vite/src/components/TransferSummary.jsx


In [37]:
%%writefile react-query-practice/vite/src/styles/app.css

body {
  margin: 0;
  font-family: Arial, sans-serif;
  background-color: #f5f6f8;
}

.container {
  max-width: 960px;
  margin: 0 auto;
  padding: 20px;
}

h1 {
  margin-bottom: 10px;
}

.description {
  color: #666;
  margin-bottom: 20px;
}

.panel {
  background: #ffffff;
  border-radius: 8px;
  padding: 16px;
  margin-bottom: 16px;
  box-shadow: 0 2px 6px rgba(0, 0, 0, 0.08);
}

.transfer-form {
  display: flex;
  flex-direction: column;
  gap: 10px;
}

.transfer-form input,
.transfer-form select {
  padding: 10px;
  border: 1px solid #ccc;
  border-radius: 6px;
}

.action-button {
  padding: 10px 16px;
  border: none;
  background-color: #0064ff;
  color: white;
  border-radius: 6px;
  cursor: pointer;
}

.action-button:hover {
  background-color: #004ed6;
}

.status {
  margin-top: 10px;
  color: #333;
}

.status.error {
  color: red;
}

.transfer-table {
  width: 100%;
  border-collapse: collapse;
  margin-top: 10px;
}

.transfer-table th,
.transfer-table td {
  border: 1px solid #ddd;
  padding: 8px;
  text-align: center;
}

.transfer-table th {
  background-color: #f0f2f5;
}

Overwriting react-query-practice/vite/src/styles/app.css


# React Query 기반 Query Invalidation 및 비동기 상태 UI 대응(Loading/Error)

In [10]:
%%writefile react-practice/node-server/server.js

const express = require('express');
const cors = require('cors');
const mysql = require('mysql2/promise');
const path = require('path');

const app = express();
const PORT = 3000;

app.use(cors());
app.use(express.json());

// static 디렉토리 제공
app.use(express.static(path.join(__dirname, 'static')));

// DB 연결
const pool = mysql.createPool({
  host: 'localhost',
  user: 'testuser',
  password: '1234',
  database: 'testdb',
  waitForConnections: true,
  connectionLimit: 10,
  queueLimit: 0,
  dateStrings: true
});

// 문의 목록 조회
app.get('/api/tickets', async function (req, res) {
  try {
    const [rows] = await pool.query(
      'SELECT * FROM tickets ORDER BY id DESC'
    );
    res.json(rows);
  } catch (error) {
    res.status(500).json({ message: '문의 목록 조회 실패' });
  }
});

// 문의 등록
app.post('/api/tickets', async function (req, res) {
  try {
    const { customer_name, title, content } = req.body;

    if (!customer_name || !title || !content) {
      return res.status(400).json({ message: '모든 항목을 입력해야 합니다.' });
    }

    if (title.length >= 100) {
      return res.status(400).json({ message: '제목은 100자 미만이어야 합니다.' });
    }

    if (title.includes('오류테스트')) {
      return res.status(500).json({ message: '강제 오류가 발생했습니다.' });
    }

    await pool.query(
      'INSERT INTO tickets (customer_name, title, content, status) VALUES (?, ?, ?, ?)',
      [customer_name, title, content, '접수대기']
    );

    res.json({ success: true });
  } catch (error) {
    res.status(500).json({ message: '문의 등록 실패' });
  }
});

// React 라우팅 대응
app.get('/', function (req, res) {
  res.sendFile(path.join(__dirname, 'static', 'index.html'));
});

app.get('/*rest', function (req, res) {
  res.sendFile(path.join(__dirname, 'static', 'index.html'));
});

app.listen(PORT, function () {
  console.log('server running on 3000');
});

Overwriting react-practice/node-server/server.js


In [11]:
%%writefile react-practice/react/src/main.jsx

import React from 'react';
import ReactDOM from 'react-dom/client';
import { QueryClient, QueryClientProvider } from '@tanstack/react-query';
import App from './App';
import './styles/app.css';

const queryClient = new QueryClient();

ReactDOM.createRoot(document.getElementById('root')).render(
  <React.StrictMode>
    <QueryClientProvider client={queryClient}>
      <App />
    </QueryClientProvider>
  </React.StrictMode>
);

Overwriting react-practice/react/src/main.jsx


In [18]:
%%writefile react-practice/react/src/App.jsx

import { useQuery } from '@tanstack/react-query';
import { fetchProducts } from './api/productsApi';
import ProductForm from './components/ProductForm';
import ProductSummary from './components/ProductSummary';
import ProductTable from './components/ProductTable';

function App() {
  const {
    data,
    isLoading,
    isFetching,
    error,
    dataUpdatedAt
  } = useQuery({
    queryKey: ['products'],
    queryFn: fetchProducts,
    staleTime: 10000
  });

  return (
    <div className="container">
      <h1>금융 상품 캐싱 실습</h1>
      <p className="description">
        React Query의 데이터 캐싱 전략과 Stale-While-Revalidate 동작을 확인한다.
      </p>

      <ProductForm />

      <div className="panel">
        {isLoading && (
          <p className="status">상품 목록을 최초 조회하는 중입니다.</p>
        )}

        {!isLoading && isFetching && (
          <p className="status">
            캐시된 데이터를 먼저 보여주고 최신 데이터를 다시 확인하는 중입니다.
          </p>
        )}

        {error && <p className="status error">{error.message}</p>}

        {!isLoading && dataUpdatedAt > 0 && (
          <p className="status info">
            마지막 캐시 갱신 시각: {new Date(dataUpdatedAt).toLocaleTimeString('ko-KR')}
          </p>
        )}
      </div>

      {data && !error && (
        <>
          <ProductSummary products={data} />
          <ProductTable products={data} />
        </>
      )}
    </div>
  );
}

export default App;

Overwriting react-practice/react/src/App.jsx


In [13]:
%%writefile react-practice/react/src/api/productsApi.js

export async function fetchProducts() {
  const response = await fetch('/api/products');

  if (!response.ok) {
    const errorData = await response.json();
    throw new Error(errorData.message || '상품 목록 조회 실패');
  }

  return await response.json();
}

export async function createProduct(productData) {
  const response = await fetch('/api/products', {
    method: 'POST',
    headers: {
      'Content-Type': 'application/json'
    },
    body: JSON.stringify(productData)
  });

  if (!response.ok) {
    const errorData = await response.json();
    throw new Error(errorData.message || '상품 등록 실패');
  }

  return await response.json();
}

Overwriting react-practice/react/src/api/productsApi.js


In [14]:
%%writefile react-practice/react/src/components/ProductForm.jsx

import { useState } from 'react';
import { useMutation, useQueryClient } from '@tanstack/react-query';
import { createProduct } from '../api/productsApi';

function ProductForm() {
  const queryClient = useQueryClient();

  const [form, setForm] = useState({
    product_name: '',
    category: '예금',
    price: '',
    risk_level: '낮음'
  });

  const mutation = useMutation({
    mutationFn: createProduct,

    onSuccess: function () {
      queryClient.invalidateQueries({ queryKey: ['products'] });

      setForm({
        product_name: '',
        category: '예금',
        price: '',
        risk_level: '낮음'
      });
    }
  });

  function handleChange(e) {
    setForm({
      ...form,
      [e.target.name]: e.target.value
    });
  }

  function handleSubmit(e) {
    e.preventDefault();

    mutation.mutate({
      ...form,
      price: Number(form.price)
    });
  }

  return (
    <div className="panel">
      <h2>상품 등록</h2>

      <form onSubmit={handleSubmit} className="product-form">
        <input
          type="text"
          name="product_name"
          value={form.product_name}
          onChange={handleChange}
          placeholder="상품명"
        />

        <select
          name="category"
          value={form.category}
          onChange={handleChange}
        >
          <option value="예금">예금</option>
          <option value="채권">채권</option>
          <option value="펀드">펀드</option>
        </select>

        <input
          type="number"
          name="price"
          value={form.price}
          onChange={handleChange}
          placeholder="금액"
        />

        <select
          name="risk_level"
          value={form.risk_level}
          onChange={handleChange}
        >
          <option value="낮음">낮음</option>
          <option value="중간">중간</option>
          <option value="높음">높음</option>
        </select>

        <button
          type="submit"
          className="action-button"
          disabled={mutation.isPending}
        >
          {mutation.isPending ? '등록 중...' : '상품 등록'}
        </button>
      </form>

      {mutation.isPending && (
        <p className="status">서버에 상품을 등록하는 중입니다.</p>
      )}

      {mutation.isError && (
        <p className="status error">{mutation.error.message}</p>
      )}

      {mutation.isSuccess && (
        <p className="status success">
          상품 등록이 완료되었고 캐시를 무효화하여 최신 데이터를 다시 조회했습니다.
        </p>
      )}
    </div>
  );
}

export default ProductForm;

Overwriting react-practice/react/src/components/ProductForm.jsx


In [15]:
%%writefile react-practice/react/src/components/ProductTable.jsx

function ProductTable({ products }) {
  return (
    <div className="panel">
      <h2>상품 목록</h2>

      <table className="product-table">
        <thead>
          <tr>
            <th>번호</th>
            <th>상품명</th>
            <th>분류</th>
            <th>금액</th>
            <th>위험도</th>
            <th>등록 시각</th>
          </tr>
        </thead>
        <tbody>
          {products.map(function (item) {
            return (
              <tr key={item.id}>
                <td>{item.id}</td>
                <td>{item.product_name}</td>
                <td>{item.category}</td>
                <td>{Number(item.price).toLocaleString('ko-KR')}원</td>
                <td>{item.risk_level}</td>
                <td>{item.created_at}</td>
              </tr>
            );
          })}
        </tbody>
      </table>
    </div>
  );
}

export default ProductTable;

Overwriting react-practice/react/src/components/ProductTable.jsx


In [16]:
%%writefile react-practice/react/src/components/ProductSummary.jsx

function ProductSummary({ products }) {
  const totalCount = products.length;

  const totalPrice = products.reduce(function (sum, item) {
    return sum + Number(item.price);
  }, 0);

  const highRiskCount = products.filter(function (item) {
    return item.risk_level === '높음';
  }).length;

  return (
    <div className="panel">
      <h2>상품 요약</h2>
      <p>전체 상품 수: {totalCount}개</p>
      <p>총 금액: {totalPrice.toLocaleString('ko-KR')}원</p>
      <p>고위험 상품 수: {highRiskCount}개</p>
    </div>
  );
}

export default ProductSummary;

Overwriting react-practice/react/src/components/ProductSummary.jsx


In [17]:
%%writefile react-practice/react/src/styles/app.css

body {
  margin: 0;
  font-family: Arial, sans-serif;
  background-color: #f5f6f8;
}

.container {
  max-width: 1100px;
  margin: 0 auto;
  padding: 20px;
}

h1 {
  margin-bottom: 10px;
}

.description {
  color: #666;
  margin-bottom: 20px;
}

.panel {
  background: #ffffff;
  border-radius: 8px;
  padding: 16px;
  margin-bottom: 16px;
  box-shadow: 0 2px 6px rgba(0, 0, 0, 0.08);
}

.product-form {
  display: flex;
  flex-direction: column;
  gap: 10px;
}

.product-form input,
.product-form select {
  padding: 10px;
  border: 1px solid #ccc;
  border-radius: 6px;
  font-size: 14px;
}

.action-button {
  padding: 10px 16px;
  border: none;
  background-color: #0064ff;
  color: white;
  border-radius: 6px;
  cursor: pointer;
}

.action-button:hover {
  background-color: #004ed6;
}

.action-button:disabled {
  background-color: #9ebcf7;
  cursor: not-allowed;
}

.status {
  margin-top: 10px;
  color: #333;
}

.status.error {
  color: red;
}

.status.success {
  color: #006400;
}

.status.info {
  color: #1f4f99;
}

.product-table {
  width: 100%;
  border-collapse: collapse;
  margin-top: 10px;
}

.product-table th,
.product-table td {
  border: 1px solid #ddd;
  padding: 8px;
  text-align: center;
}

.product-table th {
  background-color: #f0f2f5;
}

Overwriting react-practice/react/src/styles/app.css


# React Query 기반 데이터 캐싱 전략 및 Stale-While-Revalidate

In [22]:
%%writefile reactWms/node-server/server.js

const express = require('express');
const cors = require('cors');
const mysql = require('mysql2/promise');
const path = require('path');

const app = express();
const PORT = 3000;

app.use(cors());
app.use(express.json());

// static 디렉토리 제공
app.use(express.static(path.join(__dirname, 'static')));

// DB 연결
const pool = mysql.createPool({
  host: 'localhost',
  user: 'testuser',
  password: '1234',
  database: 'testdb',
  waitForConnections: true,
  connectionLimit: 10,
  queueLimit: 0,
  dateStrings: true
});

// 상품 목록 조회
app.get('/api/products', async function (req, res) {
  try {
    const [rows] = await pool.query(
      'SELECT * FROM products ORDER BY id DESC'
    );

    await new Promise(function (resolve) {
      setTimeout(resolve, 2000);
    });

    res.json(rows);
  } catch (error) {
    res.status(500).json({ message: '상품 목록 조회 실패' });
  }
});

// 상품 등록
app.post('/api/products', async function (req, res) {
  try {
    const { product_name, category, price, risk_level } = req.body;

    if (!product_name || !category || !price || !risk_level) {
      return res.status(400).json({ message: '모든 항목을 입력해야 합니다.' });
    }

    await pool.query(
      'INSERT INTO products (product_name, category, price, risk_level) VALUES (?, ?, ?, ?)',
      [product_name, category, price, risk_level]
    );

    res.json({ success: true });
  } catch (error) {
    res.status(500).json({ message: '상품 등록 실패' });
  }
});

// React 라우팅 대응
app.get('/', function (req, res) {
  res.sendFile(path.join(__dirname, 'static', 'index.html'));
});

app.get('/*rest', function (req, res) {
  res.sendFile(path.join(__dirname, 'static', 'index.html'));
});

app.listen(PORT, function () {
  console.log('server running on 3000');
});

Overwriting reactWms/node-server/server.js


In [14]:
%%writefile reactWms/react/src/main.jsx

import React from 'react';
import ReactDOM from 'react-dom/client';
import { QueryClient, QueryClientProvider } from '@tanstack/react-query';
import App from './App';
import './styles/app.css';

const queryClient = new QueryClient();

ReactDOM.createRoot(document.getElementById('root')).render(
  <React.StrictMode>
    <QueryClientProvider client={queryClient}>
      <App />
    </QueryClientProvider>
  </React.StrictMode>
);

Overwriting reactWms/react/src/main.jsx


In [15]:
%%writefile reactWms/react/src/App.jsx

import { useQuery } from '@tanstack/react-query';
import { fetchProducts } from './api/productsApi';
import ProductForm from './components/ProductForm';
import ProductSummary from './components/ProductSummary';
import ProductTable from './components/ProductTable';

function App() {
  const {
    data,
    isLoading,
    isFetching,
    error,
    dataUpdatedAt
  } = useQuery({
    queryKey: ['products'],
    queryFn: fetchProducts,
    staleTime: 10000
  });

  return (
    <div className="container">
      <h1>금융 상품 캐싱 실습</h1>
      <p className="description">
        React Query의 데이터 캐싱 전략과 Stale-While-Revalidate 동작을 확인한다.
      </p>

      <ProductForm />

      <div className="panel">
        {isLoading && (
          <p className="status">상품 목록을 최초 조회하는 중입니다.</p>
        )}

        {!isLoading && isFetching && (
          <p className="status">
            캐시된 데이터를 먼저 보여주고 최신 데이터를 다시 확인하는 중입니다.
          </p>
        )}

        {error && <p className="status error">{error.message}</p>}

        {!isLoading && dataUpdatedAt > 0 && (
          <p className="status info">
            마지막 캐시 갱신 시각: {new Date(dataUpdatedAt).toLocaleTimeString('ko-KR')}
          </p>
        )}
      </div>

      {data && !error && (
        <>
          <ProductSummary products={data} />
          <ProductTable products={data} />
        </>
      )}
    </div>
  );
}

export default App;

Overwriting reactWms/react/src/App.jsx


In [16]:
%%writefile reactWms/react/src/api/productsApi.js

export async function fetchProducts() {
  const response = await fetch('/api/products');

  if (!response.ok) {
    const errorData = await response.json();
    throw new Error(errorData.message || '상품 목록 조회 실패');
  }

  return await response.json();
}

export async function createProduct(productData) {
  const response = await fetch('/api/products', {
    method: 'POST',
    headers: {
      'Content-Type': 'application/json'
    },
    body: JSON.stringify(productData)
  });

  if (!response.ok) {
    const errorData = await response.json();
    throw new Error(errorData.message || '상품 등록 실패');
  }

  return await response.json();
}

Overwriting reactWms/react/src/api/productsApi.js


In [17]:
%%writefile reactWms/react/src/components/ProductForm.jsx

import { useState } from 'react';
import { useMutation, useQueryClient } from '@tanstack/react-query';
import { createProduct } from '../api/productsApi';

function ProductForm() {
  const queryClient = useQueryClient();

  const [form, setForm] = useState({
    product_name: '',
    category: '예금',
    price: '',
    risk_level: '낮음'
  });

  const mutation = useMutation({
    mutationFn: createProduct,

    onSuccess: function () {
      queryClient.invalidateQueries({ queryKey: ['products'] });

      setForm({
        product_name: '',
        category: '예금',
        price: '',
        risk_level: '낮음'
      });
    }
  });

  function handleChange(e) {
    setForm({
      ...form,
      [e.target.name]: e.target.value
    });
  }

  function handleSubmit(e) {
    e.preventDefault();

    mutation.mutate({
      ...form,
      price: Number(form.price)
    });
  }

  return (
    <div className="panel">
      <h2>상품 등록</h2>

      <form onSubmit={handleSubmit} className="product-form">
        <input
          type="text"
          name="product_name"
          value={form.product_name}
          onChange={handleChange}
          placeholder="상품명"
        />

        <select
          name="category"
          value={form.category}
          onChange={handleChange}
        >
          <option value="예금">예금</option>
          <option value="채권">채권</option>
          <option value="펀드">펀드</option>
        </select>

        <input
          type="number"
          name="price"
          value={form.price}
          onChange={handleChange}
          placeholder="금액"
        />

        <select
          name="risk_level"
          value={form.risk_level}
          onChange={handleChange}
        >
          <option value="낮음">낮음</option>
          <option value="중간">중간</option>
          <option value="높음">높음</option>
        </select>

        <button
          type="submit"
          className="action-button"
          disabled={mutation.isPending}
        >
          {mutation.isPending ? '등록 중...' : '상품 등록'}
        </button>
      </form>

      {mutation.isPending && (
        <p className="status">서버에 상품을 등록하는 중입니다.</p>
      )}

      {mutation.isError && (
        <p className="status error">{mutation.error.message}</p>
      )}

      {mutation.isSuccess && (
        <p className="status success">
          상품 등록이 완료되었고 캐시를 무효화하여 최신 데이터를 다시 조회했습니다.
        </p>
      )}
    </div>
  );
}

export default ProductForm;

Overwriting reactWms/react/src/components/ProductForm.jsx


In [18]:
%%writefile reactWms/react/src/components/ProductTable.jsx

function ProductTable({ products }) {
  return (
    <div className="panel">
      <h2>상품 목록</h2>

      <table className="product-table">
        <thead>
          <tr>
            <th>번호</th>
            <th>상품명</th>
            <th>분류</th>
            <th>금액</th>
            <th>위험도</th>
            <th>등록 시각</th>
          </tr>
        </thead>
        <tbody>
          {products.map(function (item) {
            return (
              <tr key={item.id}>
                <td>{item.id}</td>
                <td>{item.product_name}</td>
                <td>{item.category}</td>
                <td>{Number(item.price).toLocaleString('ko-KR')}원</td>
                <td>{item.risk_level}</td>
                <td>{item.created_at}</td>
              </tr>
            );
          })}
        </tbody>
      </table>
    </div>
  );
}

export default ProductTable;

Overwriting reactWms/react/src/components/ProductTable.jsx


In [19]:
%%writefile reactWms/react/src/components/ProductSummary.jsx

function ProductSummary({ products }) {
  const totalCount = products.length;

  const totalPrice = products.reduce(function (sum, item) {
    return sum + Number(item.price);
  }, 0);

  const highRiskCount = products.filter(function (item) {
    return item.risk_level === '높음';
  }).length;

  return (
    <div className="panel">
      <h2>상품 요약</h2>
      <p>전체 상품 수: {totalCount}개</p>
      <p>총 금액: {totalPrice.toLocaleString('ko-KR')}원</p>
      <p>고위험 상품 수: {highRiskCount}개</p>
    </div>
  );
}

export default ProductSummary;

Overwriting reactWms/react/src/components/ProductSummary.jsx


In [20]:
%%writefile reactWms/react/src/styles/app.css

body {
  margin: 0;
  font-family: Arial, sans-serif;
  background-color: #f5f6f8;
}

.container {
  max-width: 1100px;
  margin: 0 auto;
  padding: 20px;
}

h1 {
  margin-bottom: 10px;
}

.description {
  color: #666;
  margin-bottom: 20px;
}

.panel {
  background: #ffffff;
  border-radius: 8px;
  padding: 16px;
  margin-bottom: 16px;
  box-shadow: 0 2px 6px rgba(0, 0, 0, 0.08);
}

.product-form {
  display: flex;
  flex-direction: column;
  gap: 10px;
}

.product-form input,
.product-form select {
  padding: 10px;
  border: 1px solid #ccc;
  border-radius: 6px;
  font-size: 14px;
}

.action-button {
  padding: 10px 16px;
  border: none;
  background-color: #0064ff;
  color: white;
  border-radius: 6px;
  cursor: pointer;
}

.action-button:hover {
  background-color: #004ed6;
}

.action-button:disabled {
  background-color: #9ebcf7;
  cursor: not-allowed;
}

.status {
  margin-top: 10px;
  color: #333;
}

.status.error {
  color: red;
}

.status.success {
  color: #006400;
}

.status.info {
  color: #1f4f99;
}

.product-table {
  width: 100%;
  border-collapse: collapse;
  margin-top: 10px;
}

.product-table th,
.product-table td {
  border: 1px solid #ddd;
  padding: 8px;
  text-align: center;
}

.product-table th {
  background-color: #f0f2f5;
}

Overwriting reactWms/react/src/styles/app.css
